# 📚 Retrieval-Augmented Generation (RAG) — Document Question Answering System
### Powered by LangChain, Google Gemini API, ChromaDB & Gradio

---
## 📌 Project Overview

This project implements an end-to-end **Retrieval-Augmented Generation (RAG)** system that allows users to ask questions about a custom unstructured text document and receive answers grounded in the document content.

For this implementation, the source document is a **Harry Potter text file (`HarryPotterRag.txt`)**.

Instead of relying only on the language model's pre-trained knowledge, the system first retrieves the most relevant information from the document and then provides that information to the LLM as context for generating the final answer.

This project implements a Question-Answering RAG Pipeline over custom unstructured text documents.

By combining Google Gemini (gemini-2.5-flash / gemini-3.5-flash) with Chroma Vector Database and LangChain Expression Language (LCEL), the system retrieves only the relevant contextual chunks from the source document to generate accurate, hallucination-free responses.

### 🎯 Project Objective

The objective of this project is to build a complete document-based Question Answering pipeline that can:

- Load an unstructured text document
- Split the document into smaller semantic chunks
- Convert text chunks into numerical vector embeddings
- Store and index embeddings using ChromaDB
- Retrieve the most relevant document chunks for a user query
- Provide the retrieved context to a Large Language Model (LLM)
- Generate a context-grounded answer using Google Gemini
- Provide an interactive interface for querying the document


### 🔄 Overall RAG Pipeline

```text
HarryPotterRag.txt
        ↓
Document Loading
        ↓
Text Chunking
        ↓
Gemini Embeddings
        ↓
ChromaDB Vector Store
        ↓
Similarity Retrieval
        ↓
Relevant Context
        ↓
Prompt Template
        ↓
Google Gemini LLM
        ↓
Context-Grounded Answer
        ↓
Interactive User Interface

## 📦 Step 1: Environment Setup & Dependencies

Install and import the libraries required to build the RAG pipeline.

The implementation uses:

- **LangChain** for document processing and RAG orchestration
- **Google Gemini** for embeddings and text generation
- **ChromaDB** for vector storage and similarity search
- **Gradio** for the interactive application interface

In [6]:
import os
import gradio as gr

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import warnings
warnings.filterwarnings('ignore')

C:\Users\om020\AppData\Local\Temp\ipykernel_7512\1703738389.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


---

## 🔑 Step 2: Google Gemini API Configuration

The application uses the **Google Gemini API** for:

1. Generating vector embeddings for document chunks
2. Generating natural-language answers from retrieved context

The API key is entered securely at runtime using `getpass`, rather than being hard-coded directly into the notebook.

> **Security Note:** API keys should never be committed to public repositories such as GitHub.

In [10]:
from getpass import getpass
import os

GOOGLE_API_KEY = getpass("Enter your Google API Key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

Enter your Google API Key:  ········


In [11]:
from google import genai
import os

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

models = client.models.list()

for model in models:
    supported = getattr(model, "supported_actions", [])

    if "generateContent" in supported:
        print(model.name)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

test_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

response = test_llm.invoke(
    "Hello I'm RoleXx"
)

print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text', 'text': 'Hello RoleXx! Nice to meet you. How can I help you today?', 'extras': {'signature': 'EoIGCv8FARFNMg+j9AOEb7cYRjItv42693fH4Q6osPv4fagg+2DfYNEVy7ojC6BfzmueVpkpQCix7QfvRoHLwkEWV8KvY/wyIL1tAYWcPpPXK1bqWK6rGhSw6f78D9o6gYo5Tx1lp8hwhYcBT0DaVRjpvoan6EeSODeJnRVeJ/Y+WAEUfjaP6Iecg36PwkvR8jblnuxpF7vzjaw/PtD4iTui0kSIo9YOW9WOMFDyzkoEkNVTBBWJ21B8dm/kaVGvz6P1cwNF5+KwXZqvINHHyDxfVKfHXugoy/VOo5xvcu9UYlSiElxEECGzB3kgud3AP8b2DabnN9U7FE4GpgSeVODS0/S2u7OvqHPZT4GtFwMaaYhGJiAt3+o/7yTMD+JkwM5hJRIa8dHMScYiys25Uz8ASdIzqWwk29T6KWjop3/6g2exnMtpp2ZlqZJNZ/rHQB/U60QzNd4RHYe/gyKsaMIWxXmNIFXPCxEqZ92MTNLRvYxip6gKO4nYE8ukmnHnZyMIfW3I/aaw3Y/BAVHrjL7sJB/iW/tgr1+auqtKQDSEvBgxWZllV7FQUlliS0Fi7/IwlFP5hqPI260JCAtLLIuGXsM8ITgjllYlltNOyYrW+PqQ6xuJlctr+jerxJOYkbnLfyG9ztiSYfbbgVr7uj1t8pjgNARcwLqrOE66iQqHw/etnsyQ7axvUpTc1MPws+jMRge0KymW02Mqfh0ht2Idr172YvB46wPA577nvbKKsHUVupd/eDy3+DqMOlUGES5LY+Rm+AUuY2idO5WOauf4FTUqGc4Z4JeP9gJ2knoDt4FytkgLHJ/GjF5nGw+DPUcrJjbqKI40XCQSoad1UYgjG8S8VmUu/MmafdMDb4LkgBi49K2NyAOl

In [13]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

test_embedding = embeddings.embed_query(
    "Who is Harry Potter?"
)

print("Embedding generated successfully.")
print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Embedding generated successfully.
Embedding dimensions: 3072
First 10 values: [0.011348297, 0.03477306, 0.018061483, -0.019972349, 0.008695679, -0.018019227, -0.010890927, 0.010897345, 0.015121684, -0.045104057]


---
## 📄 Step 3: Document Ingestion & Text Splitting
Load the raw dataset (`HarryPotterRag.txt`) and split the text into smaller, manageable chunks with overlap. 

* **`chunk_size=500`**: Maximum character length per chunk.
* **`chunk_overlap=50`**: Overlapping characters between consecutive chunks to preserve contextual flow.

In [14]:
def load_and_split(filepath):

    loader = TextLoader(
        filepath,
        encoding="utf-8"
    )

    docs = loader.load()

    print("Splitting Data into Chunks")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    splits = splitter.split_documents(docs)

    print(f"Split {len(splits)} chunks")

    return splits

In [15]:
#Jupyter_Book/2026 Python prac/08 August Py_ML/HarryPotterRag.tx
file_path = "HarryPotterRag.txt"

In [16]:
splits = load_and_split(file_path)

Splitting Data into Chunks
Split 4 chunks


In [17]:
splits

[Document(metadata={'source': 'HarryPotterRag.txt'}, page_content='Harry James Potter is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. Harry was born on July 31, 1980, to James Potter and Lily Potter (née Evans).'),
 Document(metadata={'source': 'HarryPotterRag.txt'}, page_content="When Harry was just one year old, Lord Voldemort attacked the Potter family in Godric's Hollow. Lord Voldemort killed Harry's parents, James and Lily Potter. However, when Voldemort attempted to cast the Killing Curse (Avada Kedavra) on baby Harry, Lily's sacrificial protection caused the curse to rebound, destroying Voldemort's physical form and leaving Harry with a distinctive lightning bolt-shaped scar on his forehead."),
 Document(metadata={'source': 'HarryPotterRag.txt'}, page_content="Harry grew up unaware of his magical heritage with his non-magical relatives, the Dursleys: his Aunt Petunia, Uncle Vernon, and cousin Dudley. On his eleventh birthday, Rubeus Hagrid

In [18]:
print("Number of chunks:", len(splits))

print("\nFirst chunk:")
print(splits[0].page_content)

print("\nMetadata:")
print(splits[0].metadata)

Number of chunks: 4

First chunk:
Harry James Potter is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. Harry was born on July 31, 1980, to James Potter and Lily Potter (née Evans).

Metadata:
{'source': 'HarryPotterRag.txt'}


In [21]:
for i in splits:
    print(i)
    print("\n\n-----------\n\n")

page_content='Harry James Potter is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. Harry was born on July 31, 1980, to James Potter and Lily Potter (née Evans).' metadata={'source': 'HarryPotterRag.txt'}


-----------


page_content='When Harry was just one year old, Lord Voldemort attacked the Potter family in Godric's Hollow. Lord Voldemort killed Harry's parents, James and Lily Potter. However, when Voldemort attempted to cast the Killing Curse (Avada Kedavra) on baby Harry, Lily's sacrificial protection caused the curse to rebound, destroying Voldemort's physical form and leaving Harry with a distinctive lightning bolt-shaped scar on his forehead.' metadata={'source': 'HarryPotterRag.txt'}


-----------


page_content='Harry grew up unaware of his magical heritage with his non-magical relatives, the Dursleys: his Aunt Petunia, Uncle Vernon, and cousin Dudley. On his eleventh birthday, Rubeus Hagrid delivered Harry's acceptance letter to Hogwart

---

## 🧠 Step 4: Vector Embeddings & ChromaDB Indexing

The document chunks are converted into numerical vector representations using **Google Gemini Embeddings**.

These vectors capture the semantic meaning of the text and allow the system to compare a user's question with document content based on semantic similarity.

The generated embeddings are stored in **ChromaDB**, which acts as the vector database.

### Retrieval Flow

```text
Document Chunk
      ↓
Gemini Embedding Model
      ↓
Vector Representation
      ↓
ChromaDB
      ↓
Similarity Search
      ↓
Relevant Document Chunks

In [22]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="harry_potter_rag"
)

print("Chroma Vector Store created successfully.")

Chroma Vector Store created successfully.


In [23]:
retriever = vectorstore.as_retriever()

print("Retriever created successfully.")

Retriever created successfully.


In [24]:
question = "Who killed Harry's parents?"

retrieved_docs = retriever.invoke(question)

print("Number of retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):

    print("\n" + "=" * 80)
    print(f"RETRIEVED DOCUMENT {i + 1}")
    print("=" * 80)

    print(doc.page_content)

    print("\nMetadata:")
    print(doc.metadata)

Number of retrieved documents: 4

RETRIEVED DOCUMENT 1
When Harry was just one year old, Lord Voldemort attacked the Potter family in Godric's Hollow. Lord Voldemort killed Harry's parents, James and Lily Potter. However, when Voldemort attempted to cast the Killing Curse (Avada Kedavra) on baby Harry, Lily's sacrificial protection caused the curse to rebound, destroying Voldemort's physical form and leaving Harry with a distinctive lightning bolt-shaped scar on his forehead.

Metadata:
{'source': 'HarryPotterRag.txt'}

RETRIEVED DOCUMENT 2
Harry James Potter is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. Harry was born on July 31, 1980, to James Potter and Lily Potter (née Evans).

Metadata:
{'source': 'HarryPotterRag.txt'}

RETRIEVED DOCUMENT 3
Albus Dumbledore, the Headmaster of Hogwarts, served as an important mentor and protector for Harry throughout his school years. Dumbledore guided Harry in uncovering the secret of Voldemort's Horcruxes

### 🔍 Retrieval Validation

A sample question is used to validate the retriever independently from the LLM.

Example:

> **Who killed Harry's parents?**

The retriever searches the ChromaDB vector store and returns the document chunks that are semantically most relevant to the question.

This separates the **retrieval stage** from the **generation stage**, making it easier to verify whether the RAG system is retrieving useful context before passing it to the LLM.

---

## 📝 Step 5: Prompt Template & Gemini LLM

The retrieved document chunks are provided to the Gemini LLM as context.

The prompt is intentionally designed to instruct the model to answer **only from the retrieved context**.

### Prompt Strategy

```text
Retrieved Context
        +
User Question
        ↓
Prompt Template
        ↓
Gemini LLM
        ↓
Context-Grounded Answer

In [25]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.5
)

print("LLM created successfully.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LLM created successfully.


In [29]:
from langchain_core.prompts import PromptTemplate

# 1. Define the prompt template
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

In [30]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

print("Document formatting function created successfully.")

Document formatting function created successfully.


In [31]:
# 2. Prepare inputs and invoke
question = "Who killed Harry's parents?"
context = format_docs(retrieved_docs)

formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(formatted_prompt)

text="Answer the question based only on the following context:\n\nWhen Harry was just one year old, Lord Voldemort attacked the Potter family in Godric's Hollow. Lord Voldemort killed Harry's parents, James and Lily Potter. However, when Voldemort attempted to cast the Killing Curse (Avada Kedavra) on baby Harry, Lily's sacrificial protection caused the curse to rebound, destroying Voldemort's physical form and leaving Harry with a distinctive lightning bolt-shaped scar on his forehead.\n\nHarry James Potter is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. Harry was born on July 31, 1980, to James Potter and Lily Potter (née Evans).\n\nAlbus Dumbledore, the Headmaster of Hogwarts, served as an important mentor and protector for Harry throughout his school years. Dumbledore guided Harry in uncovering the secret of Voldemort's Horcruxes—magical objects containing pieces of Voldemort's soul—which needed to be destroyed to defeat him once and for all.

In [32]:
response = llm.invoke(formatted_prompt)

print(response.content)

[{'type': 'text', 'text': "Based on the provided context, Lord Voldemort killed Harry's parents.", 'extras': {'signature': 'EuUHCuIHARFNMg+ilHEm/rttUakD6LAU4AFdrYv0vdE5gBWuYZG3WMreQ7WA2LpealHSyLVUwfKf17VjqbOOuQ7jKEgUN3PqWR7kDvHRYQMot7LqNsJ3KiArTeYu/3+mw12TkOGRzCFLyZXEsjBT4rXy8ZN9Xuhv2k2obbDkMn6i/dILZYf/QnaRR7TSIXg5ioRnLHcrD0dlg6ERJryNIqLtF6nB7Un5FDSypAcPJLPg8IIzHKT/cBl+1xYb041R02kMkYZ2mf//Q2mQ+5Z7Ur1oh78Sgqgg+WJqiAhI0M1I7VJtg0CxkuvtSRjFtg9E6e63Nl04g2Tj7NY3G1PUpawC5BIV+Xnfm+JxeJ6/PFb3utw6gjSU1uGSai3vj7nlKgWklbkIyMDU6WYCc5GPElmo7+Ljd6qyESpo4/RIrxg4qyzdSX0km4+eGPydZZwIRaEaNHh/avh9Xhg1F2BK0UbsyYPRsICfLvF5zxqScuDi7pXxwEyanbbE++8VJzqIPkaupNgyGyZ6o8nzNrV6QZ1YLKQZE5kcekeUc57W6gHtEOl865/gyUPHMivJn7w4+rOJkv58TR9QCtgwg7a/cF9GXiSLWQCCgI92cUHZXFM0buxeLsT+hqjaRtR8OgWF2PDyoW+Lv4uEV6CImMuGjo2mgEO9fmc3YmNS47exdLftbL2rrlPNcYKXAjWEfwNPe9Q/Ut6qdOm0VkVnKBxLR/dP8NIv9gBU3l3MbYP6n4MFTLfUhgmBoS/YsCJdSxzhC5W+UkS2RNnKrk6qZ/kz6iNv1Ij8lkRpTAeqDJ9XkEWqtmjDcDz5ULoNlSBKfRnOoac8eYa/2BXLqsNo3Cryz7uW61HCfM+fr9BkLPmYa2HM

---
## 🔗 Step 6: Construct the LangChain (LCEL) Pipeline
Assemble the end-to-end RAG pipeline using LangChain Expression Language (LCEL):

$$\text{User Query} \longrightarrow \text{Retriever + Formatter} \longrightarrow \text{Prompt Template} \longrightarrow \text{LLM} \longrightarrow \text{String Parser}$$

In [33]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


---
## 🧪 Step 7: Pipeline Verification & Query Testing
Test the RAG chain locally with sample domain-specific questions and edge cases (out-of-context questions) to verify retrieval precision.

In [35]:
message = input("Ask question: ")

output = rag_chain.invoke(message)

print("\nHelpful Answer:")
print(output)

Ask question:  Who are Harry's closest friends?



Helpful Answer:
Based on the provided context, Harry's close friends are Ron Weasley and Hermione Granger.


In [36]:
message = input("Ask question: ")

output = rag_chain.invoke(message)

print("\nAnswer:")
print(output)

Ask question:  Who was Harry's mentor at Hogwarts?



Answer:
Based on the provided context, Harry's mentor at Hogwarts was Albus Dumbledore.


In [37]:
message = input("Ask question: ")

output = rag_chain.invoke(message)

print("\nHelpful Answer:")
print(output)

Ask question:  Who is Harry



Helpful Answer:
Based on the provided context, Harry (Harry James Potter) is a wizard and the central figure in the fight against the dark wizard Lord Voldemort. 

Additionally, the context notes that he:
* Is the son of James and Lily Potter.
* Grew up with his non-magical relatives, the Dursleys.
* Is a Hogwarts student sorted into Gryffindor house, a skilled Quidditch Seeker, and close friends with Ron Weasley and Hermione Granger.
* Survived Lord Voldemort's Killing Curse as a baby due to his mother's sacrificial protection, which left him with a lightning bolt-shaped scar on his forehead.
* Was mentored by Hogwarts Headmaster Albus Dumbledore to find and destroy Voldemort's Horcruxes.


In [38]:
message = input("Ask question: ")

output = rag_chain.invoke(message)

print("\nAnswer:")
print(output)

Ask question:  What is the capital of India?



Answer:
Based on the provided context, there is no information about the capital of India.


---

## 📊 Test Results & Observations

The RAG pipeline successfully demonstrated the following behavior:

| Test | Expected Behavior | Result |
|---|---|---|
| Document-based question | Retrieve relevant context and answer correctly | ✅ Passed |
| Semantic question | Retrieve relevant related chunks | ✅ Passed |
| Out-of-context question | Avoid unsupported information | ✅ Passed |

### Key Observation

The system is able to retrieve relevant information from `HarryPotterRag.txt` and use that retrieved context to generate an answer through Gemini.

The out-of-context test also demonstrates the importance of grounding the LLM response in retrieved document context.

---
## 🚀 Step 8: Deploy Interactive Chat Interface (Gradio)
Launch a local and shareable web application using Gradio's `ChatInterface` to interact with the RAG model in real time.

In [39]:
def respond(message, history):

    response = rag_chain.invoke(message)

    return response

In [40]:
test_response = respond(
    "Who killed Harry's parents?",
    []
)

print(test_response)

Based on the provided context, Lord Voldemort killed Harry's parents.


In [41]:
demo = gr.ChatInterface(
    fn=respond,
    textbox=gr.Textbox(
        placeholder="Ask a question regarding Harry Potter..."
    ),
    title="Harry Potter RAG Document Reader",
    description="Ask questions based on the Harry Potter document."
)

print("Gradio interface created successfully.")

Gradio interface created successfully.


In [42]:
demo.launch(
    share=True,
    debug=True
)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Keyboard interruption in main thread... closing server.


In [43]:
demo.close()

Closing server running on port: 7860


# 🎯 Project Summary

This project demonstrates an end-to-end **Retrieval-Augmented Generation (RAG)** workflow for document-based Question Answering.

### What was implemented?

- ✅ Document ingestion using `TextLoader`
- ✅ Recursive text chunking
- ✅ Gemini-based text embeddings
- ✅ ChromaDB vector storage
- ✅ Semantic document retrieval
- ✅ Context-aware prompt construction
- ✅ Gemini LLM response generation
- ✅ LangChain LCEL pipeline
- ✅ Out-of-context query testing
- ✅ Interactive question-answering interface

### Final Architecture

```text
                HarryPotterRag.txt
                        │
                        ▼
                  Document Loader
                        │
                        ▼
                  Text Chunking
                        │
                        ▼
                Gemini Embeddings
                        │
                        ▼
                    ChromaDB
                        │
                        ▼
                    Retriever
                        │
              ┌─────────┴─────────┐
              │                   │
        User Question       Retrieved Context
              │                   │
              └─────────┬─────────┘
                        ▼
                  Prompt Template
                        │
                        ▼
                   Gemini LLM
                        │
                        ▼
                    Answer